In [1]:
# ── Path setup — works from ANY subfolder ──
import sys
from pathlib import Path

cwd = Path.cwd()
# ابحث عن جذر المشروع = الفولدر اللي فيه "fortyguard"
PROJECT_ROOT = cwd if (cwd / "fortyguard").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook location: {cwd}")
print(f"PROJECT_ROOT:      {PROJECT_ROOT}")

Notebook location: d:\heat-to-shelf\src
PROJECT_ROOT:      d:\heat-to-shelf


In [2]:
from pathlib import Path
src_dir = Path("d:/temperature-api-quickstart/src")

print("ملفات الـ init الموجودة:")
for f in src_dir.glob("*init*"):
    print(f"  → '{f.name}'  (طول الاسم: {len(f.name)})")

# الاسم الصح لازم يكون:
correct = "__init__.py"
print(f"\nالمطلوب: '{correct}' (طول: {len(correct)})")
print(f"موجود؟ {(src_dir / correct).exists()}")

ملفات الـ init الموجودة:
  → '__init__.py'  (طول الاسم: 11)

المطلوب: '__init__.py' (طول: 11)
موجود؟ True


In [3]:
from pathlib import Path
src = Path("d:/temperature-api-quickstart/src")

old = src / "_init_.py"
new = src / "__init__.py"

if old.exists():
    old.rename(new)
    print("✅ renamed to __init__.py")
elif new.exists():
    print("✅ __init__.py already correct")
else:
    print("❌ ولا ملف موجود — هنعمله:")
    new.write_text('"""Heat-to-Shelf source package."""')
    print("✅ created __init__.py")

✅ __init__.py already correct


In [4]:
import sys
from pathlib import Path

print("═══ 1. sys.path entries ═══")
for i, p in enumerate(sys.path[:6]):
    print(f"  [{i}] {p}")

print("\n═══ 2. هل فيه src/ مكرر؟ ═══")
root = Path("d:/temperature-api-quickstart")
print(f"  src/ موجود:            {(root / 'src').exists()}")
print(f"  src/src/ موجود:        {(root / 'src' / 'src').exists()}")  # ← المتهم الرئيسي!
print(f"  src/risk_engine.py:    {(root / 'src' / 'risk_engine.py').exists()}")
print(f"  src/src/risk_engine.py: {(root / 'src' / 'src' / 'risk_engine.py').exists()}")

print("\n═══ 3. sys.modules cache ═══")
for mod in ["src", "src.risk_engine", "src.loader"]:
    print(f"  {mod}: {'محمل (cached)' if mod in sys.modules else 'غير محمل'}")

print("\n═══ 4. محاولة import مباشرة ═══")
try:
    import src
    print(f"  ✅ src imported — المكان: {src.__file__}")
except Exception as e:
    print(f"  ❌ {e}")

═══ 1. sys.path entries ═══
  [0] d:\heat-to-shelf
  [1] C:\Python314\python314.zip
  [2] C:\Python314\DLLs
  [3] C:\Python314\Lib
  [4] C:\Python314
  [5] d:\heat-to-shelf\venv

═══ 2. هل فيه src/ مكرر؟ ═══
  src/ موجود:            True
  src/src/ موجود:        True
  src/risk_engine.py:    True
  src/src/risk_engine.py: False

═══ 3. sys.modules cache ═══
  src: غير محمل
  src.risk_engine: غير محمل
  src.loader: غير محمل

═══ 4. محاولة import مباشرة ═══
  ✅ src imported — المكان: d:\heat-to-shelf\src\__init__.py


In [5]:
from pathlib import Path
dup = Path("d:/temperature-api-quickstart/src/src")

if dup.exists():
    print("المحتوى:")
    for f in dup.iterdir():
        print(f"  {f.name}")
    print("\nلو فيه ملفات قديمة مكررة → ده هو السبب")
    print("Python بيلاقي src/src/ الأول (ترتيب الـ path) وبيقف لما ميلقاش risk_engine جواه")

المحتوى:

لو فيه ملفات قديمة مكررة → ده هو السبب
Python بيلاقي src/src/ الأول (ترتيب الـ path) وبيقف لما ميلقاش risk_engine جواه


In [6]:
from pathlib import Path
src = Path("d:/temperature-api-quickstart/src")

required = ["__init__.py", "risk_engine.py", "loader.py",
            "schemas.py", "explainer.py"]
print("فحص الأصل:")
for f in required:
    exists = (src / f).exists()
    size = (src / f).stat().st_size if exists else 0
    print(f"  {'✅' if exists else '❌'} {f:<18} ({size} bytes)")

فحص الأصل:
  ✅ __init__.py        (0 bytes)
  ✅ risk_engine.py     (6375 bytes)
  ✅ loader.py          (1821 bytes)
  ✅ schemas.py         (1267 bytes)
  ✅ explainer.py       (4691 bytes)


In [7]:
import shutil
from pathlib import Path

dup = Path("d:/temperature-api-quickstart/src/src")

# تأكيد نهائي: المكرر فيه __pycache__ فقط — ولا .py واحد
py_files = list(dup.rglob("*.py"))
print(f"ملفات .py في المكرر: {len(py_files)}")

if dup.exists() and len(py_files) == 0:
    shutil.rmtree(dup)
    print("✅ تم مسح src/src/ — كان كاش فارغ فقط")
else:
    print("⚠️ فيه ملفات .py — ابعتلي القائمة قبل أي مسح!")

ملفات .py في المكرر: 0
✅ تم مسح src/src/ — كان كاش فارغ فقط


In [15]:
import sys

# نضف أي حاجة محملة باسم src
for mod in list(sys.modules):
    if mod == "src" or mod.startswith("src."):
        del sys.modules[mod]
        print(f"  نظّفت: {mod}")

# أعد المحاولة
from src.risk_engine import RiskEngine, CargoProfile
from src.loader import load_scenario, build_observations
from src.explainer import explain_assessment, explain_comparison

print("\n✅✅✅ Imports working — المشكلة خلصت!")
print(f"src.risk_engine المكان: {sys.modules['src.risk_engine'].__file__}")

  نظّفت: src

✅✅✅ Imports working — المشكلة خلصت!
src.risk_engine المكان: d:\temperature-api-quickstart\src\risk_engine.py


In [8]:
# ── Explainer test — complete fixed version ──
import sys
from pathlib import Path

# Path setup that works from src/, notebooks/, or root
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "fortyguard").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}\n")

from src.risk_engine import RiskEngine, CargoProfile
from src.loader import load_scenario, build_observations
from src.explainer import explain_assessment, explain_comparison

CACHE = PROJECT_ROOT / "cache"

engine = RiskEngine(CargoProfile(
    name="Wine (placeholder)",
    warning_threshold_c=25.0,
    critical_threshold_c=28.0,
    source_name="PLACEHOLDER", source_url="",
))

# CRITICAL case — اللحظة الأهم في الـ demo:
a16 = engine.assess(build_observations(
    load_scenario(CACHE, "2026-08-19", "16:00")))
print("="*60)
print(explain_assessment(a16, engine.cargo, trip_minutes=61))
print("="*60)

# SAFE case:
a06 = engine.assess(build_observations(
    load_scenario(CACHE, "2026-08-19", "06:00")))
print()
print(explain_assessment(a06, engine.cargo, trip_minutes=61))

# Comparison summary:
print("="*60)
results = {"06:00": a06, "16:00": a16}
print(explain_comparison(results, engine.cargo, trip_minutes=61))

PROJECT_ROOT: d:\heat-to-shelf

Cargo: Wine (placeholder) — warning at 25.0°C, critical at 28.0°C.
Trip duration: 61 minutes.

🚨 VERDICT: DO NOT DEPART at this hour.

The critical override fired: 20 route segments reached or exceeded the critical threshold (28.0°C). The shipment spent 38.8 minutes above the warning threshold, including one continuous stretch of 20.0 minutes.

The weighted score was 83.5/100, but the override rule treats any critical-threshold breach as disqualifying — a brief breach is still a breach.

Recommendation: choose an earlier departure scenario; 06:00 measured 0 minutes of threshold exposure on this date.

Cargo: Wine (placeholder) — warning at 25.0°C, critical at 28.0°C.
Trip duration: 61 minutes.

✅ VERDICT: Safe to depart.

Peak route temperature stayed below the warning threshold (25.0°C) for the entire 61-minute trip. Zero minutes of threshold exposure were measured.

Score: 0.0/100 (SAFE).
Across 2 departure scenarios, 06:00 carries the lowest thermal e